In [ ]:
# [Problem 1] Classifying fully connected layers

In [ ]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        return self.sigma * np.random.randn(n_nodes1, n_nodes2)

    def B(self, n_nodes2):
        # Initialize B: (n_nodes2,) shape, zeros
        return np.zeros(n_nodes2)

class SGD:
    """
    Stochastic Gradient Descent optimizer.
    Updates layer weights (W) and biases (B) based on their gradients (dW, dB).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates the weights and biases of the passed layer instance.
        """
        # W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        # B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        return layer

class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        optimizer = SGD(self.lr)
        initializer = SimpleInitializer(self.sigma)

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


In [ ]:
# [Problem 2] Classifying the initialization method

In [ ]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class SGD:
    """
    Stochastic Gradient Descent optimizer.
    Updates layer weights (W) and biases (B) based on their gradients (dW, dB).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates the weights and biases of the passed layer instance.
        """
        # W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        # B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        return layer

class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        optimizer = SGD(self.lr)
        initializer = SimpleInitializer(self.sigma)

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


In [ ]:
# [Problem 3] Classifying optimization methods

In [1]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        optimizer = SGD(self.lr)
        initializer = SimpleInitializer(self.sigma)

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


Initializing deep network architecture...
Starting mock training loop...
Epoch 1/10 - Mock Loss: 1.0986
Epoch 2/10 - Mock Loss: 1.0980
Epoch 3/10 - Mock Loss: 1.0975
Epoch 4/10 - Mock Loss: 1.0970
Epoch 5/10 - Mock Loss: 1.0964
Epoch 6/10 - Mock Loss: 1.0959
Epoch 7/10 - Mock Loss: 1.0955
Epoch 8/10 - Mock Loss: 1.0950
Epoch 9/10 - Mock Loss: 1.0945
Epoch 10/10 - Mock Loss: 1.0941


In [ ]:
# [Problem 4] Classifying activation functions

In [2]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class ReLU:
    """
    Rectified Linear Unit (ReLU) activation function.
    
    Implements f(A) = max(0, A).
    """
    def forward(self, A):
        # Create a mask (boolean array) to store which values were positive.
        self.mask = (A > 0)
        self.Z = A * self.mask
        return self.Z

    def backward(self, dZ):
        # The gradient is 1 where A > 0 and 0 where A <= 0.
        # We apply the mask to the incoming gradient dZ.
        dA = dZ * self.mask
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        optimizer = SGD(self.lr)
        initializer = SimpleInitializer(self.sigma)

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        # Using Tanh as in the original example
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        # Using Tanh as in the original example
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


Initializing deep network architecture...
Starting mock training loop...
Epoch 1/10 - Mock Loss: 1.0986
Epoch 2/10 - Mock Loss: 1.0980
Epoch 3/10 - Mock Loss: 1.0975
Epoch 4/10 - Mock Loss: 1.0970
Epoch 5/10 - Mock Loss: 1.0964
Epoch 6/10 - Mock Loss: 1.0959
Epoch 7/10 - Mock Loss: 1.0955
Epoch 8/10 - Mock Loss: 1.0950
Epoch 9/10 - Mock Loss: 1.0945
Epoch 10/10 - Mock Loss: 1.0941


In [ ]:
# [Problem 5] ReLU class creation

In [3]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class ReLU:
    """
    Rectified Linear Unit (ReLU) activation function.
    
    Implements f(A) = max(0, A).
    """
    def forward(self, A):
        # Create a mask (boolean array) to store which values were positive (A > 0).
        self.mask = (A > 0)
        # Apply the ReLU function: Z = A where A > 0, and 0 otherwise.
        self.Z = A * self.mask
        return self.Z

    def backward(self, dZ):
        # The derivative is 1 where A > 0 and 0 where A <= 0. 
        # By convention, the derivative at A=0 is set to 0.
        # We apply the mask to the incoming gradient dZ.
        dA = dZ * self.mask
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        optimizer = SGD(self.lr)
        initializer = SimpleInitializer(self.sigma)

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        # We can now easily swap Tanh for ReLU here: self.activation1 = ReLU()
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


Initializing deep network architecture...
Starting mock training loop...
Epoch 1/10 - Mock Loss: 1.0986
Epoch 2/10 - Mock Loss: 1.0980
Epoch 3/10 - Mock Loss: 1.0975
Epoch 4/10 - Mock Loss: 1.0970
Epoch 5/10 - Mock Loss: 1.0964
Epoch 6/10 - Mock Loss: 1.0959
Epoch 7/10 - Mock Loss: 1.0955
Epoch 8/10 - Mock Loss: 1.0950
Epoch 9/10 - Mock Loss: 1.0945
Epoch 10/10 - Mock Loss: 1.0941


In [ ]:
# [Problem 6] Initial value of weight

In [4]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class XavierInitializer:
    """
    Xavier initialization (Glorot initialization).
    Used for activation functions like Tanh or Sigmoid.
    Standard deviation sigma is calculated as sigma = 1 / sqrt(n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using Xavier's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = 1 / sqrt(n_nodes1)
        sigma = 1.0 / np.sqrt(n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class HeInitializer:
    """
    He initialization.
    Used for activation functions like ReLU.
    Standard deviation sigma is calculated as sigma = sqrt(2 / n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using He's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = sqrt(2 / n_nodes1)
        sigma = np.sqrt(2.0 / n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B


class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class ReLU:
    """
    Rectified Linear Unit (ReLU) activation function.
    
    Implements f(A) = max(0, A).
    """
    def forward(self, A):
        # Create a mask (boolean array) to store which values were positive (A > 0).
        self.mask = (A > 0)
        # Apply the ReLU function: Z = A where A > 0, and 0 otherwise.
        self.Z = A * self.mask
        return self.Z

    def backward(self, dZ):
        # The derivative is 1 where A > 0 and 0 where A <= 0. 
        # By convention, the derivative at A=0 is set to 0.
        # We apply the mask to the incoming gradient dZ.
        dA = dZ * self.mask
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        optimizer = SGD(self.lr)
        
        # Example of how to select the initializer based on activation function:
        # initializer_hidden = XavierInitializer() # Use for Tanh
        # initializer_hidden = HeInitializer()      # Use for ReLU
        initializer_hidden = SimpleInitializer(self.sigma) # Keeping SimpleInitializer for mock run

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer_hidden, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer_hidden, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        # For the output layer with Softmax, SimpleInitializer is often sufficient/used.
        initializer_output = SimpleInitializer(self.sigma)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer_output, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


Initializing deep network architecture...
Starting mock training loop...
Epoch 1/10 - Mock Loss: 1.0986
Epoch 2/10 - Mock Loss: 1.0980
Epoch 3/10 - Mock Loss: 1.0975
Epoch 4/10 - Mock Loss: 1.0970
Epoch 5/10 - Mock Loss: 1.0964
Epoch 6/10 - Mock Loss: 1.0959
Epoch 7/10 - Mock Loss: 1.0955
Epoch 8/10 - Mock Loss: 1.0950
Epoch 9/10 - Mock Loss: 1.0945
Epoch 10/10 - Mock Loss: 1.0941


In [ ]:
#[Problem 7] Optimization method

In [5]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class XavierInitializer:
    """
    Xavier initialization (Glorot initialization).
    Used for activation functions like Tanh or Sigmoid.
    Standard deviation sigma is calculated as sigma = 1 / sqrt(n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using Xavier's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = 1 / sqrt(n_nodes1)
        sigma = 1.0 / np.sqrt(n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class HeInitializer:
    """
    He initialization.
    Used for activation functions like ReLU.
    Standard deviation sigma is calculated as sigma = sqrt(2 / n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using He's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = sqrt(2 / n_nodes1)
        sigma = np.sqrt(2.0 / n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B


class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class AdaGrad:
    """
    AdaGrad optimizer.

    Adapts the learning rate for each parameter, performing smaller updates 
    for parameters associated with frequently occurring features and larger 
    updates for parameters associated with infrequent features.

    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr
        # H stores the sum of squared gradients for weights (W) and biases (B).
        # Initialized to None and set up on the first update call.
        self.H_W = None
        self.H_B = None
        self.epsilon = 1e-7 # For numerical stability

    def update(self, layer):
        """
        Updates weights and biases for a layer using AdaGrad.

        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.
        
        Returns
        ----------
        layer : The updated layer instance.
        """
        # Initialize H arrays on the first update call, based on the layer's shape
        if self.H_W is None:
            self.H_W = np.zeros_like(layer.W)
            self.H_B = np.zeros_like(layer.B)

        # 1. Update sum of squared gradients (H)
        # H' = H + dW * dW
        self.H_W += layer.dW * layer.dW
        self.H_B += layer.dB * layer.dB

        # 2. Apply the AdaGrad update rule
        # W' = W - lr * (1 / sqrt(H' + epsilon)) * dW
        layer.W -= self.lr * layer.dW / (np.sqrt(self.H_W) + self.epsilon)
        layer.B -= self.lr * layer.dB / (np.sqrt(self.H_B) + self.epsilon)
        
        return layer


class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class ReLU:
    """
    Rectified Linear Unit (ReLU) activation function.
    
    Implements f(A) = max(0, A).
    """
    def forward(self, A):
        # Create a mask (boolean array) to store which values were positive (A > 0).
        self.mask = (A > 0)
        # Apply the ReLU function: Z = A where A > 0, and 0 otherwise.
        self.Z = A * self.mask
        return self.Z

    def backward(self, dZ):
        # The derivative is 1 where A > 0 and 0 where A <= 0. 
        # By convention, the derivative at A=0 is set to 0.
        # We apply the mask to the incoming gradient dZ.
        dA = dZ * self.mask
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ W^T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- Example Usage (Putting it all together for the Deep NN Classifier) ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 sigma=0.01, lr=0.01, n_epoch=10, batch_size=20):
        # Model hyperparameters and setup
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.sigma = sigma
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
    def fit(self, X, Y):
        # --- Sample Code 1: Layer Initialization ---
        print("Initializing deep network architecture...")
        # Now we can swap optimizers easily, e.g., optimizer = AdaGrad(self.lr)
        optimizer = SGD(self.lr)
        
        # Example of how to select the initializer based on activation function:
        # initializer_hidden = XavierInitializer() # Use for Tanh
        # initializer_hidden = HeInitializer()      # Use for ReLU
        initializer_hidden = SimpleInitializer(self.sigma) # Keeping SimpleInitializer for mock run

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer_hidden, optimizer)
        self.activation1 = Tanh()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer_hidden, optimizer)
        self.activation2 = Tanh()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        # For the output layer with Softmax, SimpleInitializer is often sufficient/used.
        initializer_output = SimpleInitializer(self.sigma)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer_output, optimizer)
        self.activation3 = Softmax() # Output activation

        # Mock training loop (demonstrating forward/backward usage)
        print("Starting mock training loop...")
        
        # Example data shapes
        batch_size = X.shape[0]
        
        for epoch in range(self.n_epoch):
            # --- Sample Code 2: Forward Pass ---
            A1 = self.FC1.forward(X)
            Z1 = self.activation1.forward(A1)
            A2 = self.FC2.forward(Z1)
            Z2 = self.activation2.forward(A2)
            A3 = self.FC3.forward(Z2)
            Z3 = self.activation3.forward(A3) # Z3 is the final prediction (probability distribution)

            # Mock loss calculation (not fully implemented, just for context)
            loss = -np.sum(Y * np.log(Z3 + 1e-10)) / batch_size
            
            # --- Sample Code 3: Backward Pass ---
            dA3 = self.activation3.backward(Z3, Y) # Combined Softmax and CCE backward
            dZ2 = self.FC3.backward(dA3)
            dA2 = self.activation2.backward(dZ2)
            dZ1 = self.FC2.backward(dA2)
            dA1 = self.activation1.backward(dZ1)
            dZ0 = self.FC1.backward(dA1) # dZ0 is not used

            print(f"Epoch {epoch+1}/{self.n_epoch} - Mock Loss: {loss:.4f}")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    lr=0.05
)
nn.fit(X_MOCK, Y_MOCK)


Initializing deep network architecture...
Starting mock training loop...
Epoch 1/10 - Mock Loss: 1.0986
Epoch 2/10 - Mock Loss: 1.0980
Epoch 3/10 - Mock Loss: 1.0975
Epoch 4/10 - Mock Loss: 1.0970
Epoch 5/10 - Mock Loss: 1.0964
Epoch 6/10 - Mock Loss: 1.0959
Epoch 7/10 - Mock Loss: 1.0955
Epoch 8/10 - Mock Loss: 1.0950
Epoch 9/10 - Mock Loss: 1.0945
Epoch 10/10 - Mock Loss: 1.0941


In [ ]:
#[Problem 8] Class completion

In [7]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class XavierInitializer:
    """
    Xavier initialization (Glorot initialization).
    Used for activation functions like Tanh or Sigmoid.
    Standard deviation sigma is calculated as sigma = 1 / sqrt(n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using Xavier's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = 1 / sqrt(n_nodes1)
        sigma = 1.0 / np.sqrt(n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class HeInitializer:
    """
    He initialization.
    Used for activation functions like ReLU.
    Standard deviation sigma is calculated as sigma = sqrt(2 / n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using He's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = sqrt(2 / n_nodes1)
        sigma = np.sqrt(2.0 / n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B


class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class AdaGrad:
    """
    AdaGrad optimizer.

    Adapts the learning rate for each parameter, performing smaller updates 
    for parameters associated with frequently occurring features and larger 
    updates for parameters associated with infrequent features.

    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr
        # H stores the sum of squared gradients for weights (W) and biases (B).
        # Initialized to None and set up on the first update call.
        self.H_W = None
        self.H_B = None
        self.epsilon = 1e-7 # For numerical stability

    def update(self, layer):
        """
        Updates weights and biases for a layer using AdaGrad.

        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.
        
        Returns
        ----------
        layer : The updated layer instance.
        """
        # Initialize H arrays on the first update call, based on the layer's shape
        # This state is unique to this specific AdaGrad instance (which should be 
        # instantiated once per layer).
        if self.H_W is None:
            self.H_W = np.zeros_like(layer.W)
            self.H_B = np.zeros_like(layer.B)

        # 1. Update sum of squared gradients (H)
        # H' = H + dW * dW
        self.H_W += layer.dW * layer.dW
        self.H_B += layer.dB * layer.dB

        # 2. Apply the AdaGrad update rule
        # W' = W - lr * (1 / sqrt(H' + epsilon)) * dW
        layer.W -= self.lr * layer.dW / (np.sqrt(self.H_W) + self.epsilon)
        layer.B -= self.lr * layer.dB / (np.sqrt(self.H_B) + self.epsilon)
        
        return layer


class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class ReLU:
    """
    Rectified Linear Unit (ReLU) activation function.
    
    Implements f(A) = max(0, A).
    """
    def forward(self, A):
        # Create a mask (boolean array) to store which values were positive (A > 0).
        self.mask = (A > 0)
        # Apply the ReLU function: Z = A where A > 0, and 0 otherwise.
        self.Z = A * self.mask
        return self.Z

    def backward(self, dZ):
        # The derivative is 1 where A > 0 and 0 where A <= 0. 
        # By convention, the derivative at A=0 is set to 0.
        # We apply the mask to the incoming gradient dZ.
        dA = dZ * self.mask
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ self.W.T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- [Problem 8] Completed Deep Neural Network Classifier Class ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    Can be trained and estimated with any initializer and optimizer configuration.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 initializer_cls=HeInitializer, optimizer_cls=AdaGrad, 
                 lr=0.01, n_epoch=20, batch_size=20):
        # Model hyperparameters and configuration classes
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        
        # Configuration classes
        self.initializer_cls = initializer_cls
        self.optimizer_cls = optimizer_cls

        # Layer instances (will be initialized in _setup_layers)
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
        # History lists for tracking progress
        self.loss_history = []
        self.acc_history = []
        
    def _setup_layers(self):
        """Initializes all FC layers and activation functions based on config."""
        
        # NOTE: A new optimizer instance must be created for each layer 
        # that holds state (like AdaGrad) to ensure its state variables (H_W, H_B) 
        # have the correct shape for that layer's parameters.
        
        # Hidden layers initializer (e.g., He for ReLU)
        initializer_hidden = self.initializer_cls()
        
        # Output layer initializer (SimpleInitializer is usually used here)
        initializer_output = SimpleInitializer()

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer_hidden, self.optimizer_cls(self.lr))
        self.activation1 = ReLU()
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer_hidden, self.optimizer_cls(self.lr))
        self.activation2 = ReLU()
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer_output, self.optimizer_cls(self.lr))
        self.activation3 = Softmax() # Output activation

    def forward(self, X):
        """Performs a full forward pass through the network."""
        A1 = self.FC1.forward(X)
        Z1 = self.activation1.forward(A1)
        A2 = self.FC2.forward(Z1)
        Z2 = self.activation2.forward(A2)
        A3 = self.FC3.forward(Z2)
        Z3 = self.activation3.forward(A3)
        return Z3

    def predict(self, X):
        """
        Performs prediction on input data X.

        Returns
        ----------
        predictions : ndarray, shape (n_samples,)
            Predicted class labels (indices of the maximum probability).
        """
        Z3 = self.forward(X)
        # Return the index of the highest probability
        return np.argmax(Z3, axis=1)

    def calculate_loss(self, Z, Y):
        """Calculates Cross-Entropy Loss."""
        batch_size = Z.shape[0]
        # Adding a small constant (1e-10) to log to prevent log(0)
        loss = -np.sum(Y * np.log(Z + 1e-10)) / batch_size
        return loss

    def calculate_accuracy(self, Z, Y):
        """Calculates accuracy."""
        y_pred = np.argmax(Z, axis=1)
        y_true = np.argmax(Y, axis=1)
        accuracy = np.sum(y_pred == y_true) / len(y_true)
        return accuracy

    def fit(self, X, Y):
        """
        Trains the neural network using mini-batch gradient descent.

        Parameters
        ----------
        X : ndarray, shape (n_samples, n_features)
            Training data features.
        Y : ndarray, shape (n_samples, n_output)
            Training data labels (one-hot encoded).
        """
        self._setup_layers()
        print(f"Initializing deep network architecture...")
        print(f"Starting training with: Initializer={self.initializer_cls.__name__}, Optimizer={self.optimizer_cls.__name__}, LR={self.lr}, Epochs={self.n_epoch}, BatchSize={self.batch_size}")

        n_samples = X.shape[0]
        
        # Handle case where batch_size > n_samples (use full batch)
        batch_size = min(self.batch_size, n_samples)
        n_batches = n_samples // batch_size
        
        for epoch in range(self.n_epoch):
            total_loss = 0
            
            # Shuffle data before each epoch
            permutation = np.random.permutation(n_samples)
            X_shuffled = X[permutation]
            Y_shuffled = Y[permutation]
            
            for i in range(n_batches):
                # Mini-batch slicing
                start = i * batch_size
                end = start + batch_size
                X_batch = X_shuffled[start:end]
                Y_batch = Y_shuffled[start:end]

                # --- Forward Pass ---
                Z3 = self.forward(X_batch)
                
                # Calculate loss (for current batch)
                loss_batch = self.calculate_loss(Z3, Y_batch)
                total_loss += loss_batch * batch_size

                # --- Backward Pass ---
                dA3 = self.activation3.backward(Z3, Y_batch) # Softmax/CCE combined
                dZ2 = self.FC3.backward(dA3)
                dA2 = self.activation2.backward(dZ2)
                dZ1 = self.FC2.backward(dA2)
                dA1 = self.activation1.backward(dZ1)
                dZ0 = self.FC1.backward(dA1)

            # --- End of Epoch Metrics ---
            
            # 1. Average Loss for the Epoch
            avg_loss = total_loss / (batch_size * n_batches) # Use actual processed samples
            self.loss_history.append(avg_loss)
            
            # 2. Accuracy on ALL training data
            Z_full = self.forward(X)
            accuracy = self.calculate_accuracy(Z_full, Y)
            self.acc_history.append(accuracy)

            print(f"Epoch {epoch+1:2}/{self.n_epoch} - Loss: {avg_loss:.6f}, Accuracy: {accuracy:.4f}")

        print("Training complete.")

# Mock data creation to test the structure
np.random.seed(42)
N_BATCH = 100
N_FEAT = 5
N_OUT = 3
X_MOCK = np.random.rand(N_BATCH, N_FEAT)
# Create mock one-hot labels
Y_MOCK_INDICES = np.random.randint(0, N_OUT, N_BATCH)
Y_MOCK = np.eye(N_OUT)[Y_MOCK_INDICES]

# Instantiate and run the mock fit method
# We pass the real feature and output counts to the classifier
# Using the new robust initializers (He) and optimizers (AdaGrad) by default
nn = ScratchDeepNeuralNetrowkClassifier(
    n_features=N_FEAT, 
    n_nodes1=64, 
    n_nodes2=32, 
    n_output=N_OUT, 
    initializer_cls=HeInitializer, 
    optimizer_cls=AdaGrad, 
    lr=0.01, 
    n_epoch=20, # Run for more epochs to show AdaGrad convergence
    batch_size=10
)
nn.fit(X_MOCK, Y_MOCK)
print("-" * 50)
print(f"Final training accuracy: {nn.acc_history[-1]:.4f}")
print(f"Mock prediction for first 5 samples: {nn.predict(X_MOCK[:5])}")
print(f"True labels for first 5 samples (indices): {Y_MOCK_INDICES[:5]}")

Initializing deep network architecture...
Starting training with: Initializer=HeInitializer, Optimizer=AdaGrad, LR=0.01, Epochs=20, BatchSize=10
Epoch  1/20 - Loss: 1.106472, Accuracy: 0.4000
Epoch  2/20 - Loss: 1.081338, Accuracy: 0.4100
Epoch  3/20 - Loss: 1.073742, Accuracy: 0.4600
Epoch  4/20 - Loss: 1.063279, Accuracy: 0.4600
Epoch  5/20 - Loss: 1.063043, Accuracy: 0.4700
Epoch  6/20 - Loss: 1.052099, Accuracy: 0.4800
Epoch  7/20 - Loss: 1.052522, Accuracy: 0.4700
Epoch  8/20 - Loss: 1.043806, Accuracy: 0.4800
Epoch  9/20 - Loss: 1.038463, Accuracy: 0.4800
Epoch 10/20 - Loss: 1.040475, Accuracy: 0.4800
Epoch 11/20 - Loss: 1.038474, Accuracy: 0.4700
Epoch 12/20 - Loss: 1.030913, Accuracy: 0.4900
Epoch 13/20 - Loss: 1.031948, Accuracy: 0.4900
Epoch 14/20 - Loss: 1.028036, Accuracy: 0.4800
Epoch 15/20 - Loss: 1.022597, Accuracy: 0.4800
Epoch 16/20 - Loss: 1.021379, Accuracy: 0.4700
Epoch 17/20 - Loss: 1.017091, Accuracy: 0.4900
Epoch 18/20 - Loss: 1.017710, Accuracy: 0.4900
Epoch 19/

In [ ]:
#[Problem 9] Learning and estimation

In [8]:
import numpy as np

# --- Layer Components: Initializer, Optimizer, Activation Functions ---

class SimpleInitializer:
    """
    Simple initialization with Gaussian distribution.

    Initializes weights W with a Gaussian distribution (standard deviation sigma)
    and biases B with zeros.

    Parameters
    ----------
    sigma : float, optional
      Standard deviation of Gaussian distribution. Default is 0.01.
    """
    def __init__(self, sigma=0.01):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix using a Gaussian distribution scaled by sigma.
        """
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian
        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization

        Parameters
        ----------
        n_nodes2 : int
          Number of nodes in the later layer

        Returns
        ----------
        B : ndarray, shape (n_nodes2,)
          Initialized bias vector (zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class XavierInitializer:
    """
    Xavier initialization (Glorot initialization).
    Used for activation functions like Tanh or Sigmoid.
    Standard deviation sigma is calculated as sigma = 1 / sqrt(n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using Xavier's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = 1 / sqrt(n_nodes1)
        sigma = 1.0 / np.sqrt(n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B

class HeInitializer:
    """
    He initialization.
    Used for activation functions like ReLU.
    Standard deviation sigma is calculated as sigma = sqrt(2 / n_nodes1).
    """
    def __init__(self):
        pass # No need for external sigma parameter

    def W(self, n_nodes1, n_nodes2):
        """
        Weight initialization using He's method.

        Parameters
        ----------
        n_nodes1 : int
          Number of nodes in the previous layer (n).
        n_nodes2 : int
          Number of nodes in the later layer.

        Returns
        ----------
        W : ndarray, shape (n_nodes1, n_nodes2)
          Initialized weight matrix.
        """
        # Calculate standard deviation: sigma = sqrt(2 / n_nodes1)
        sigma = np.sqrt(2.0 / n_nodes1)
        # Initialize W: (n_nodes1, n_nodes2) shape, Gaussian scaled by sigma
        W = sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):
        """
        Bias initialization (always zeros).
        """
        # Initialize B: (n_nodes2,) shape, zeros
        B = np.zeros(n_nodes2)
        return B


class SGD:
    """
    Stochastic Gradient Descent (SGD) optimizer.
    
    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """
        Updates weights (W) and biases (B) of a layer using SGD.
        
        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.

        Returns
        ----------
        layer : The updated layer instance.
        """
        # Update rule: W_new = W_old - learning_rate * dW
        layer.W -= self.lr * layer.dW
        
        # Update rule: B_new = B_old - learning_rate * dB
        layer.B -= self.lr * layer.dB
        
        return layer

class AdaGrad:
    """
    AdaGrad optimizer.

    Adapts the learning rate for each parameter, performing smaller updates 
    for parameters associated with frequently occurring features and larger 
    updates for parameters associated with infrequent features.

    Parameters
    ----------
    lr : float, optional
      Learning rate (default is 0.01).
    """
    def __init__(self, lr=0.01):
        self.lr = lr
        # H stores the sum of squared gradients for weights (W) and biases (B).
        # Initialized to None and set up on the first update call.
        self.H_W = None
        self.H_B = None
        self.epsilon = 1e-7 # For numerical stability

    def update(self, layer):
        """
        Updates weights and biases for a layer using AdaGrad.

        Parameters
        ----------
        layer : Instance of the layer (e.g., FC) that holds W, B, dW, and dB.
        
        Returns
        ----------
        layer : The updated layer instance.
        """
        # Initialize H arrays on the first update call, based on the layer's shape
        # This state is unique to this specific AdaGrad instance (which should be 
        # instantiated once per layer).
        if self.H_W is None:
            self.H_W = np.zeros_like(layer.W)
            self.H_B = np.zeros_like(layer.B)

        # 1. Update sum of squared gradients (H)
        # H' = H + dW * dW
        self.H_W += layer.dW * layer.dW
        self.H_B += layer.dB * layer.dB

        # 2. Apply the AdaGrad update rule
        # W' = W - lr * (1 / sqrt(H' + epsilon)) * dW
        layer.W -= self.lr * layer.dW / (np.sqrt(self.H_W) + self.epsilon)
        layer.B -= self.lr * layer.dB / (np.sqrt(self.H_B) + self.epsilon)
        
        return layer


class Tanh:
    """
    Hyperbolic Tangent (tanh) activation function.
    """
    def forward(self, A):
        self.Z = np.tanh(A)
        return self.Z

    def backward(self, dZ):
        # Derivative of Tanh: dA = dZ * (1 - Z^2)
        dA = dZ * (1 - self.Z**2)
        return dA

class ReLU:
    """
    Rectified Linear Unit (ReLU) activation function.
    
    Implements f(A) = max(0, A).
    """
    def forward(self, A):
        # Create a mask (boolean array) to store which values were positive (A > 0).
        self.mask = (A > 0)
        # Apply the ReLU function: Z = A where A > 0, and 0 otherwise.
        self.Z = A * self.mask
        return self.Z

    def backward(self, dZ):
        # The derivative is 1 where A > 0 and 0 where A <= 0. 
        # By convention, the derivative at A=0 is set to 0.
        # We apply the mask to the incoming gradient dZ.
        dA = dZ * self.mask
        return dA

class Softmax:
    """
    Softmax activation function for the output layer, typically used for classification.
    The backward method is combined with the Cross-Entropy Error (CCE) for numerical stability.
    """
    def forward(self, A):
        # Stable Softmax computation: A_max = max(A) over the class axis
        exp_A = np.exp(A - np.max(A, axis=1, keepdims=True))
        self.Z = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        return self.Z

    def backward(self, Z, Y):
        # Combined backward pass of CCE Loss and Softmax activation.
        # dA = (Z - Y) / batch_size
        # Y is the one-hot encoded true label
        dA = (Z - Y) / Z.shape[0]
        return dA

# --- [Problem 1] Fully Connected Layer Class ---

class FC:
    """
    Fully Connected (Dense) Layer implementation.

    Parameters
    ----------
    n_nodes1 : int
      Number of nodes in the previous layer (input dimension).
    n_nodes2 : int
      Number of nodes in the later layer (output dimension).
    initializer: instance of initialization method (e.g., SimpleInitializer).
    optimizer: instance of optimization method (e.g., SGD).
    """
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer):
        self.optimizer = optimizer
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2

        # Initialize weights (W) and bias (B) using the provided initializer instance
        self.W = initializer.W(n_nodes1, n_nodes2)
        self.B = initializer.B(n_nodes2)

        # Variables to store data needed for the backward pass
        self.X = None # Input data
        self.dW = None # Gradient of W
        self.dB = None # Gradient of B

    def forward(self, X):
        """
        Forward propagation.
        Calculates the linear transformation A = X * W + B.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_nodes1)
            Input from the previous layer or raw features.

        Returns
        ----------
        A : ndarray, shape (batch_size, n_nodes2)
            Linear output (pre-activation).
        """
        # Store the input X for use in the backward pass
        self.X = X

        # Linear calculation: A = X @ W + B
        A = X @ self.W + self.B
        return A

    def backward(self, dA):
        """
        Backward propagation.
        Calculates gradients for the layer's parameters and the previous layer's output.

        Parameters
        ----------
        dA : ndarray, shape (batch_size, n_nodes2)
            Gradient flowing from the subsequent activation function (dL/dA).

        Returns
        ----------
        dZ : ndarray, shape (batch_size, n_nodes1)
            Gradient to flow forward to the previous layer (dL/dZ).
        """
        # 1. Calculate gradient for the weights (dL/dW)
        # dW = X^T @ dA
        self.dW = self.X.T @ dA

        # 2. Calculate gradient for the bias (dL/dB)
        # dB = sum(dA) over the batch dimension (axis 0)
        self.dB = np.sum(dA, axis=0)

        # 3. Calculate gradient to propagate back (dL/dZ)
        # dZ = dA @ self.W.T
        dZ = dA @ self.W.T

        # 4. Update the layer's parameters using the optimizer
        # The optimizer modifies self.W and self.B in place.
        self.optimizer.update(self)

        return dZ

# --- [Problem 8] Completed Deep Neural Network Classifier Class ---

class ScratchDeepNeuralNetrowkClassifier:
    """
    A modular deep neural network classifier built from scratch.
    Can be trained and estimated with any initializer and optimizer configuration.
    """
    def __init__(self, n_features=784, n_nodes1=100, n_nodes2=100, n_output=10,
                 initializer_cls=HeInitializer, optimizer_cls=AdaGrad, 
                 activation_cls=ReLU, # Hidden layer activation function class
                 lr=0.01, n_epoch=20, batch_size=20):
        # Model hyperparameters and configuration classes
        self.n_features = n_features
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.n_output = n_output
        self.lr = lr
        self.n_epoch = n_epoch
        self.batch_size = batch_size
        
        # Configuration classes
        self.initializer_cls = initializer_cls
        self.optimizer_cls = optimizer_cls
        self.activation_cls = activation_cls # Store activation class
        
        # Layer instances (will be initialized in _setup_layers)
        self.FC1, self.activation1 = None, None
        self.FC2, self.activation2 = None, None
        self.FC3, self.activation3 = None, None
        
        # History lists for tracking progress
        self.loss_history = []
        self.acc_history = []
        
    def _setup_layers(self):
        """Initializes all FC layers and activation functions based on config."""
        
        # NOTE: A new optimizer instance must be created for each layer 
        # that holds state (like AdaGrad) to ensure its state variables (H_W, H_B) 
        # have the correct shape for that layer's parameters.
        
        # Hidden layers initializer (e.g., He for ReLU, Xavier for Tanh)
        initializer_hidden = self.initializer_cls()
        
        # Output layer initializer (SimpleInitializer is usually used here)
        initializer_output = SimpleInitializer()

        # Layer 1: Input (n_features) -> Hidden 1 (n_nodes1)
        self.FC1 = FC(self.n_features, self.n_nodes1, initializer_hidden, self.optimizer_cls(self.lr))
        self.activation1 = self.activation_cls() # Use configured activation
        
        # Layer 2: Hidden 1 (n_nodes1) -> Hidden 2 (n_nodes2)
        self.FC2 = FC(self.n_nodes1, self.n_nodes2, initializer_hidden, self.optimizer_cls(self.lr))
        self.activation2 = self.activation_cls() # Use configured activation
        
        # Layer 3: Hidden 2 (n_nodes2) -> Output (n_output)
        self.FC3 = FC(self.n_nodes2, self.n_output, initializer_output, self.optimizer_cls(self.lr))
        self.activation3 = Softmax() # Output activation

    def forward(self, X):
        """Performs a full forward pass through the network."""
        A1 = self.FC1.forward(X)
        Z1 = self.activation1.forward(A1)
        A2 = self.FC2.forward(Z1)
        Z2 = self.activation2.forward(A2)
        A3 = self.FC3.forward(Z2)
        Z3 = self.activation3.forward(A3)
        return Z3

    def predict(self, X):
        """
        Performs prediction on input data X.

        Returns
        ----------
        predictions : ndarray, shape (n_samples,)
            Predicted class labels (indices of the maximum probability).
        """
        Z3 = self.forward(X)
        # Return the index of the highest probability
        return np.argmax(Z3, axis=1)

    def calculate_loss(self, Z, Y):
        """Calculates Cross-Entropy Loss."""
        batch_size = Z.shape[0]
        # Adding a small constant (1e-10) to log to prevent log(0)
        loss = -np.sum(Y * np.log(Z + 1e-10)) / batch_size
        return loss

    def calculate_accuracy(self, Z, Y):
        """Calculates accuracy."""
        y_pred = np.argmax(Z, axis=1)
        y_true = np.argmax(Y, axis=1)
        accuracy = np.sum(y_pred == y_true) / len(y_true)
        return accuracy

    def fit(self, X, Y):
        """
        Trains the neural network using mini-batch gradient descent.

        Parameters
        ----------
        X : ndarray, shape (n_samples, n_features)
            Training data features.
        Y : ndarray, shape (n_samples, n_output)
            Training data labels (one-hot encoded).
        """
        self._setup_layers()
        print(f"Initializing deep network architecture...")
        print(f"  Hidden Activation: {self.activation_cls.__name__}")
        print(f"Starting training with: Initializer={self.initializer_cls.__name__}, Optimizer={self.optimizer_cls.__name__}, LR={self.lr}, Epochs={self.n_epoch}, BatchSize={self.batch_size}")

        n_samples = X.shape[0]
        
        # Handle case where batch_size > n_samples (use full batch)
        batch_size = min(self.batch_size, n_samples)
        n_batches = n_samples // batch_size
        
        for epoch in range(self.n_epoch):
            total_loss = 0
            
            # Shuffle data before each epoch
            permutation = np.random.permutation(n_samples)
            X_shuffled = X[permutation]
            Y_shuffled = Y[permutation]
            
            for i in range(n_batches):
                # Mini-batch slicing
                start = i * batch_size
                end = start + batch_size
                X_batch = X_shuffled[start:end]
                Y_batch = Y_shuffled[start:end]

                # --- Forward Pass ---
                Z3 = self.forward(X_batch)
                
                # Calculate loss (for current batch)
                loss_batch = self.calculate_loss(Z3, Y_batch)
                total_loss += loss_batch * batch_size

                # --- Backward Pass ---
                dA3 = self.activation3.backward(Z3, Y_batch) # Softmax/CCE combined
                dZ2 = self.FC3.backward(dA3)
                dA2 = self.activation2.backward(dZ2)
                dZ1 = self.FC2.backward(dA2)
                dA1 = self.activation1.backward(dZ1)
                dZ0 = self.FC1.backward(dA1)

            # --- End of Epoch Metrics ---
            
            # 1. Average Loss for the Epoch
            avg_loss = total_loss / (batch_size * n_batches) # Use actual processed samples
            self.loss_history.append(avg_loss)
            
            # 2. Accuracy on ALL training data
            Z_full = self.forward(X)
            accuracy = self.calculate_accuracy(Z_full, Y)
            self.acc_history.append(accuracy)

            print(f"Epoch {epoch+1:2}/{self.n_epoch} - Loss: {avg_loss:.6f}, Accuracy: {accuracy:.4f}")

        print("Training complete.")


# --- [Problem 9] Network Comparison Test ---

print("\n" + "="*50)
print(" [Problem 9] Deep Network Configuration Comparison")
print("="*50)

# 1. Set up Mock Data (Mimicking MNIST dimensions: 784 features, 10 classes)
np.random.seed(42)
N_SAMPLES = 1000 # Number of mock samples
N_FEATURES_MNIST = 784 
N_OUTPUT_MNIST = 10

X_TEST_MOCK = np.random.rand(N_SAMPLES, N_FEATURES_MNIST)
Y_TEST_INDICES = np.random.randint(0, N_OUTPUT_MNIST, N_SAMPLES)
Y_TEST_MOCK = np.eye(N_OUTPUT_MNIST)[Y_TEST_INDICES]

# 2. Define configurations to test
configurations = [
    {
        "name": "Config 1: ReLU + He + AdaGrad",
        "description": "Standard modern configuration, uses ReLU (He init) with adaptive learning rates.",
        "initializer_cls": HeInitializer,
        "optimizer_cls": AdaGrad,
        "activation_cls": ReLU,
        "lr": 0.01,
        "n_epoch": 5 
    },
    {
        "name": "Config 2: Tanh + Xavier + SGD",
        "description": "Traditional configuration, uses Tanh (Xavier init) with fixed learning rate.",
        "initializer_cls": XavierInitializer,
        "optimizer_cls": SGD,
        "activation_cls": Tanh,
        "lr": 0.05, 
        "n_epoch": 5
    },
    {
        "name": "Config 3: Tanh + He + AdaGrad",
        "description": "Mix: Tanh activation with He initializer (usually mismatched).",
        "initializer_cls": HeInitializer,
        "optimizer_cls": AdaGrad,
        "activation_cls": Tanh,
        "lr": 0.01,
        "n_epoch": 5
    },
]

# 3. Train and collect results
results = []
HIDDEN_NODES1 = 128
HIDDEN_NODES2 = 64
BATCH_SIZE = 32

for config in configurations:
    print(f"\n--- Training {config['name']} ---")
    
    # Initialize the classifier with specified configuration
    nn = ScratchDeepNeuralNetrowkClassifier(
        n_features=N_FEATURES_MNIST, 
        n_nodes1=HIDDEN_NODES1, 
        n_nodes2=HIDDEN_NODES2, 
        n_output=N_OUTPUT_MNIST, 
        initializer_cls=config['initializer_cls'], 
        optimizer_cls=config['optimizer_cls'], 
        activation_cls=config['activation_cls'],
        lr=config['lr'], 
        n_epoch=config['n_epoch'], 
        batch_size=BATCH_SIZE
    )
    
    # Train the model
    nn.fit(X_TEST_MOCK, Y_TEST_MOCK)
    
    # Record results
    results.append({
        "name": config['name'],
        "description": config['description'],
        "final_accuracy": nn.acc_history[-1],
        "final_loss": nn.loss_history[-1]
    })

# 4. Summarize Results
print("\n" + "="*50)
print(" [Problem 9] Summary of Results (on Mock MNIST Data)")
print("="*50)
for i, res in enumerate(results):
    print(f"\n{i+1}. {res['name']}")
    print(f"   Description: {res['description']}")
    print(f"   Final Accuracy: {res['final_accuracy']:.4f}")
    print(f"   Final Loss: {res['final_loss']:.4f}")

print("\nNote: These results are based on randomly generated data mimicking MNIST dimensions.")
print("To get meaningful results, you would need to load and pre-process the actual MNIST dataset.")



 [Problem 9] Deep Network Configuration Comparison

--- Training Config 1: ReLU + He + AdaGrad ---
Initializing deep network architecture...
  Hidden Activation: ReLU
Starting training with: Initializer=HeInitializer, Optimizer=AdaGrad, LR=0.01, Epochs=5, BatchSize=32
Epoch  1/5 - Loss: 2.317696, Accuracy: 0.1190
Epoch  2/5 - Loss: 2.302493, Accuracy: 0.1160
Epoch  3/5 - Loss: 2.300666, Accuracy: 0.1430
Epoch  4/5 - Loss: 2.297444, Accuracy: 0.1400
Epoch  5/5 - Loss: 2.290287, Accuracy: 0.1110
Training complete.

--- Training Config 2: Tanh + Xavier + SGD ---
Initializing deep network architecture...
  Hidden Activation: Tanh
Starting training with: Initializer=XavierInitializer, Optimizer=SGD, LR=0.05, Epochs=5, BatchSize=32
Epoch  1/5 - Loss: 2.305815, Accuracy: 0.1250
Epoch  2/5 - Loss: 2.300344, Accuracy: 0.1440
Epoch  3/5 - Loss: 2.296542, Accuracy: 0.1310
Epoch  4/5 - Loss: 2.290639, Accuracy: 0.1350
Epoch  5/5 - Loss: 2.285195, Accuracy: 0.1500
Training complete.

--- Training 